# Continuous Maze: controlled transfer experiments

Start with `EXPERIMENT = "temperature_carry"`. Scratch and the first sequential task start at alpha 0.1; later tasks and independent recovery inherit the final temperature of the model they start from. Actor/critic equations, rewards and full budgets otherwise match the previous factorised-actor run.

Use `matched_reset` as the control. Test `scaled_regularization` and `policy_retention` separately after the temperature comparison. These two options each extend `temperature_carry`; they do not combine both changes. Existing result notebooks are preserved.

The package must be installed in this kernel (`uv sync --locked --extra notebooks --group dev`). Restart the kernel after pulling code changes. See `docs/transfer-experiments.md` for equations and interpretation.

In [ ]:
import os
from pathlib import Path
from uuid import uuid4

import torch

from tbtrl.experiments.config import load_config
from tbtrl.experiments.diagnostics import NotebookDiagnostics
from tbtrl.experiments.plots import plot_comparison, plot_run
from tbtrl.experiments.runner import run_experiment
from tbtrl.experiments.transfer import configure_transfer

root = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "configs/tbtrl_gridworldmaze_sac_factorised_actor.json").exists()
)
# historical_reset | matched_reset | temperature_carry | scaled_regularization | policy_retention
EXPERIMENT = "temperature_carry"
# baseline | factorised_only | mlp_replay | factorised_tbtrl
ACTOR_VARIANT = "factorised_tbtrl"
SMOKE = False
DEVICE = (
    "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
)
# Run all phases. Set ("scratch", "sequential") to omit recovery.
MODES = ("scratch", "sequential", "recovery")
config = configure_transfer(
    load_config(root / "configs/tbtrl_gridworldmaze_sac_factorised_actor.json"),
    EXPERIMENT,
    ACTOR_VARIANT,
)
if SMOKE:
    config = config.smoke()
output_root = Path(os.environ.get("TBTRL_OUTPUT_ROOT", str(root / "runs")))
output = output_root / f"{config.name}-{uuid4().hex[:8]}"
print("SMOKE CHECK" if SMOKE else "FULL RESEARCH RUN")
print(f"Experiment={EXPERIMENT}; actor={ACTOR_VARIANT}; device={DEVICE}")
print(f"Seeds={config.seeds}; goals={config.goals}; temperature={config.entropy_transfer}")
print(
    f"Steps/goal={config.training['total_steps']:,}; warmup={config.training['warmup_steps']:,}; recovery={config.recovery['total_steps']:,}"
)
print(
    f"Initial alpha={config.training['initial_alpha']}; actor options={config.training.get('actor_tbtrl')}"
)
print(
    f"Transfer probes at steps {config.training['transfer_probe_steps']} and every regular evaluation."
)
get_ipython().run_line_magic("matplotlib", "inline")

In [ ]:
diagnostics = NotebookDiagnostics(rollout_episodes=config.eval_episodes)
records = run_experiment(
    config, output, device=DEVICE, modes=MODES, on_task_end=diagnostics, verbose=True
)
print(f"Saved to {output}")

In [ ]:
# Read the effective values actually used, including carried temperature.
for row in records:
    if "initial_alpha" in row:
        print(
            f"{row['mode']:10s} goal={row['task_id']} alpha={row['initial_alpha']:.6g} -> {row['final_alpha']:.6g}; steps={row['steps']:,}; solved={row['solved']}"
        )
figure = plot_run(output)
figure

In [ ]:
comparison = plot_comparison(output)
comparison

Compare `temperature_carry` against `matched_reset` with the same actor and seeds first. Early probes do not count toward stopping; the original threshold/patience and evaluation frequency still determine completion. Probe step zero has no training interactions or updates.

The two optional follow-ups are experimental: `scaled_regularization` targets covariance 0.125 I at rank 16, while `policy_retention` adds 0.01 times KL(previous policy || current policy) on selected old replay states. Neither guarantees improvement. Gradient cosine is zero when either gradient vanishes, including tasks without replay.